In [6]:
import pandas as pd
import tensorflow as tf
import sklearn

In [7]:
from tensorflow.keras.layers import Dense, Dropout, Activation, Input
from tensorflow.keras.models import Model
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

In [8]:
base = pd.read_csv('/home/beuren/Documentos/LAMIA-Bootcamp/card-14/data/regressão múltiplas saídas/games.csv')
base

,Name,Platform,Year_of_Release,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales,Critic_Score,Critic_Count,User_Score,User_Count,Developer,Rating
0,Wii Sports,Wii,2006.0,Sports,Nintendo,41.36,28.96,3.77,8.45,82.53,76.0,51.0,8,322.0,Nintendo,E
1,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24,NaN,NaN,NaN,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.68,12.76,3.79,3.29,35.52,82.0,73.0,8.3,709.0,Nintendo,E
3,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.61,10.93,3.28,2.95,32.77,80.0,73.0,8,192.0,Nintendo,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16714,Samurai Warriors: Sanada Maru,PS3,2016.0,Action,Tecmo Koei,0.00,0.00,0.01,0.00,0.01,NaN,NaN,NaN,NaN,NaN,NaN
16715,LMA Manager 2007,X360,2006.0,Sports,Codemasters,0.00,0.01,0.00,0.00,0.01,NaN,NaN,NaN,NaN,NaN,NaN
16716,Haitaka no Psychedelica,PSV,2016.0,Adventure,Idea Factory,0.00,0.00,0.01,0.00,0.01,NaN,NaN,NaN,NaN,NaN,NaN
16717,Spirits & Spells,GBA,2003.0,Platform,Wanadoo,0.01,0.00,0.00,0.00,0.01,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
base = base.drop('Other_Sales', axis = 1)
base = base.drop('Global_Sales', axis = 1)
base = base.drop('Developer', axis = 1)
# tira colunas que nao tem importancia para a predição


In [10]:
base.shape
# dimensoes do dataset

(16719, 13)

In [11]:
base.isnull().sum()
# conta os valores nulos do dataset

Name                  2
Platform              0
Year_of_Release     269
Genre                 2
Publisher            54
NA_Sales              0
EU_Sales              0
JP_Sales              0
Critic_Score       8582
Critic_Count       8582
User_Score         6704
User_Count         9129
Rating             6769
dtype: int64

In [12]:
base = base.dropna(axis = 0)
# tira todas as linhas que tem nulos

In [13]:
base.shape

(6825, 13)

In [14]:
base.isnull().sum()

Name               0
Platform           0
Year_of_Release    0
Genre              0
Publisher          0
NA_Sales           0
EU_Sales           0
JP_Sales           0
Critic_Score       0
Critic_Count       0
User_Score         0
User_Count         0
Rating             0
dtype: int64

In [15]:
base['Name'].value_counts()
# nome é proprio nao mostra nenhum padrao para o modelo

Name
Madden NFL 07                                   8
Need for Speed: Most Wanted                     8
LEGO Star Wars II: The Original Trilogy         8
Terraria                                        7
Madden NFL 08                                   7
                                               ..
Brain Age: Train Your Brain in Minutes a Day    1
Wii Fit                                         1
Wii Sports Resort                               1
Mario Kart Wii                                  1
Wii Sports                                      1
Name: count, Length: 4377, dtype: int64

In [16]:
base = base.drop('Name', axis = 1)

In [17]:
base.shape

(6825, 12)

In [18]:
base.columns

Index(['Platform', 'Year_of_Release', 'Genre', 'Publisher', 'NA_Sales',
       'EU_Sales', 'JP_Sales', 'Critic_Score', 'Critic_Count', 'User_Score',
       'User_Count', 'Rating'],
      dtype='object')

In [19]:
x = base.iloc[:, [0, 1, 2, 3, 7, 8, 9, 10, 11]].values
# pega as features caracteristicas
x

array([['Wii', 2006.0, 'Sports', ..., '8', 322.0, 'E'],
       ['Wii', 2008.0, 'Racing', ..., '8.3', 709.0, 'E'],
       ['Wii', 2009.0, 'Sports', ..., '8', 192.0, 'E'],
       ...,
       ['PC', 2014.0, 'Action', ..., '7.6', 412.0, 'M'],
       ['PC', 2011.0, 'Shooter', ..., '5.8', 43.0, 'T'],
       ['PC', 2011.0, 'Strategy', ..., '7.2', 13.0, 'E10+']],
      shape=(6825, 9), dtype=object)

In [20]:
y_na = base.iloc[:, 4].values
y_eu = base.iloc[:, 5].values
y_jp = base.iloc[:, 6].values
# 3 targets, valores de 3 regioes diferentes

In [21]:
y_na

array([4.136e+01, 1.568e+01, 1.561e+01, ..., 0.000e+00, 1.000e-02,
       0.000e+00], shape=(6825,))

In [22]:
y_eu

array([2.896e+01, 1.276e+01, 1.093e+01, ..., 1.000e-02, 0.000e+00,
       1.000e-02], shape=(6825,))

In [23]:
y_jp

array([3.77, 3.79, 3.28, ..., 0.  , 0.  , 0.  ], shape=(6825,))

In [24]:
base['Platform'].value_counts()

Platform
PS2     1140
X360     858
PS3      769
PC       651
XB       565
Wii      479
DS       464
PSP      390
GC       348
PS4      239
GBA      237
XOne     159
3DS      155
PS       150
PSV      118
WiiU      89
DC        14
Name: count, dtype: int64

In [25]:
base.columns

Index(['Platform', 'Year_of_Release', 'Genre', 'Publisher', 'NA_Sales',
       'EU_Sales', 'JP_Sales', 'Critic_Score', 'Critic_Count', 'User_Score',
       'User_Count', 'Rating'],
      dtype='object')

In [26]:
onehotencoder = ColumnTransformer(transformers=[("OneHot", OneHotEncoder(), [0, 2, 3, 8])], remainder='passthrough')
#transforma as colunas categoricas em codigos numeros sem distinção de peso
x = onehotencoder.fit_transform(x).toarray()

In [27]:
x.shape

(6825, 303)

In [28]:
x[0]

array([0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       1.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 0.000e+00, 1.000e+00, 0.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 

In [29]:
camada_entrada = Input(shape = (303,))
camada_oculta1 = Dense(units = 153, activation='relu')(camada_entrada)
camada_oculta2 = Dense(units = 153, activation='relu')(camada_oculta1)
camada_saida1 = Dense(units = 1, activation='linear')(camada_oculta2)
camada_saida2 = Dense(units = 1, activation='linear')(camada_oculta2)
camada_saida3 = Dense(units = 1, activation='linear')(camada_oculta2)
# mesmo estilo de rede mas agora com tres beuronios de saida

E0000 00:00:1786057368.396397  313968 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [30]:
regressor = Model(inputs = camada_entrada, outputs = [camada_saida1, camada_saida2, camada_saida3])
# define para o modelo as camadas de entrada e as de saida

In [31]:
regressor.compile(optimizer='adam', loss = 'mse')

In [32]:
regressor.fit(x, [y_na, y_eu, y_jp], epochs=500, batch_size=100)
# treina o modelo com os dados

Epoch 1/500
69/69 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - dense_2_loss: 323.0204 - dense_3_loss: 262.4039 - dense_4_loss: 2621.4001 - loss: 3241.9575
Epoch 2/500
69/69 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - dense_2_loss: 2.0374 - dense_3_loss: 1.5812 - dense_4_loss: 6.4467 - loss: 10.1588
Epoch 3/500
69/69 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - dense_2_loss: 1.4530 - dense_3_loss: 0.9016 - dense_4_loss: 0.9627 - loss: 3.3339
Epoch 4/500
69/69 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - dense_2_loss: 1.3822 - dense_3_loss: 0.8060 - dense_4_loss: 0.6405 - loss: 2.8495
Epoch 5/500
69/69 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - dense_2_loss: 1.2242 - dense_3_loss: 0.7148 - dense_4_loss: 0.4329 - loss: 2.3722
Epoch 6/500
69/69 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - dense_2_loss: 2.8283 - dense_3_loss: 2.1210 - dense_4_loss: 1.6196 - loss: 6.6262
Epoch 7/500
69/69 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - dense_2_loss: 2.1927 - dense_3_loss: 1.0604 - dense_4_loss: 0.9350 - loss: 4.2286
Epoch 8/500
69/69 ━━━━━━━━━━━━━━━━━━━━ 1s 8m

In [33]:
previsao_na, previsao_eu, previsao_jp = regressor.predict(x)
# faz a previsao para as tres regioes diferentes

214/214 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step


In [34]:
previsao_na, previsao_na.mean()
# pega a media das previsoes

(array([[ 4.120672  ],
        [ 4.4306326 ],
        [ 2.9992406 ],
        ...,
        [-0.10106865],
        [ 0.03711891],
        [ 0.00548643]], shape=(6825, 1), dtype=float32),
 np.float32(0.37974963))

In [35]:
y_na, y_na.mean()
# media dos resultados reais

(array([4.136e+01, 1.568e+01, 1.561e+01, ..., 0.000e+00, 1.000e-02,
        0.000e+00], shape=(6825,)),
 np.float64(0.3944835164835165))

In [36]:
from sklearn.metrics import mean_absolute_error

In [37]:
mean_absolute_error(y_na, previsao_na)

np.float64(0.27618707966909306)

In [38]:
previsao_eu, previsao_eu.mean()

(array([[2.6346996 ],
        [2.742901  ],
        [2.0304513 ],
        ...,
        [0.06073973],
        [0.05529076],
        [0.02873655]], shape=(6825, 1), dtype=float32),
 np.float32(0.20968403))

In [39]:
y_eu, y_eu.mean()

(array([2.896e+01, 1.276e+01, 1.093e+01, ..., 1.000e-02, 0.000e+00,
        1.000e-02], shape=(6825,)),
 np.float64(0.23608937728937732))

In [40]:
mean_absolute_error(y_eu, previsao_eu)

np.float64(0.18547427972603195)

In [41]:
previsao_jp, previsao_jp.mean()

(array([[ 0.67719626],
        [ 0.61230874],
        [ 0.5867939 ],
        ...,
        [-0.06524406],
        [ 0.00294908],
        [-0.00793142]], shape=(6825, 1), dtype=float32),
 np.float32(0.030699117))

In [42]:
y_jp, y_jp.mean()

(array([3.77, 3.79, 3.28, ..., 0.  , 0.  , 0.  ], shape=(6825,)),
 np.float64(0.06415824175824175))

In [43]:
mean_absolute_error(y_jp, previsao_jp)

np.float64(0.08256635826832646)